In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report


TRAIN_FILE = "records_long.json"
PROF_FILE = "subm2_labels_revealed.csv"  
OUTPUT_DIR = "results"
MODEL_DIR = "models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


def pick_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def load_json_records(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Il file {path} non è un JSON valido.\n"
            f"Errore: {e}\n"
            f"Se ti esce ancora il problema del JSONDecodeError, il file è corrotto o troncato."
        )

def read_table_flex(path):
    return pd.read_csv(path, sep=None, engine="python")

def normalize_label(x):
    if pd.isna(x):
        return x
    return str(x).strip()

# =========================
# LOAD TRAIN DATA
# =========================
data = load_json_records(TRAIN_FILE)
df = pd.DataFrame(data)

print("Colomun train:", df.columns.tolist())
print("Shape train:", df.shape)

text_col = pick_first_existing(
    df.columns,
    ["text", "Text", "content", "Content", "sentence", "Sentence"]
)

label_col = pick_first_existing(
    df.columns,
    ["final_label", "label", "Label", "model"]
)

label_id_col = pick_first_existing(
    df.columns,
    ["label_id", "Label_ID", "labelId", "LabelId"]
)

if text_col is None:
    raise ValueError("Non trovo una colonna testo nel training file.")

if label_col is None:
    raise ValueError("Non trovo una colonna label nel training file.")

df = df[[c for c in [text_col, label_col, label_id_col] if c is not None]].copy()
df = df.dropna(subset=[text_col, label_col])
df[text_col] = df[text_col].astype(str)
df[label_col] = df[label_col].astype(str).str.strip()


if label_col == "model":
    raw_to_final = {
        "chatgpt": "OpenAI",
        "openai": "OpenAI",
        "gpt": "OpenAI",
        "human": "Human",
        "gemma": "Google",
        "google": "Google",
        "llama": "Meta",
        "meta": "Meta",
        "mistral": "Anthropic",
        "claude": "Anthropic",
        "anthropic": "Anthropic",
    }
    df["target_label"] = df[label_col].str.lower().map(lambda x: raw_to_final.get(x, x))
else:
    df["target_label"] = df[label_col].astype(str).map(normalize_label)

df = df.dropna(subset=["target_label"])
df = df.drop_duplicates(subset=[text_col, "target_label"]).reset_index(drop=True)


print(df["target_label"].value_counts())


label_to_id = None
if label_id_col is not None and "final_label" in df.columns:
    tmp_map = df[["final_label", label_id_col]].dropna().drop_duplicates()
    if tmp_map["final_label"].nunique() == tmp_map[label_id_col].nunique():
        label_to_id = {
            str(row["final_label"]).strip(): int(row[label_id_col])
            for _, row in tmp_map.iterrows()
        }


encoder = LabelEncoder()
y = encoder.fit_transform(df["target_label"])
texts = df[text_col].astype(str)

print("\n final class:")
for i, c in enumerate(encoder.classes_):
    print(f"{i} -> {c}")

r
if label_to_id is None:
    label_to_id = {label: idx for idx, label in enumerate(encoder.classes_)}


X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train_texts))
print("Validation samples:", len(X_val_texts))

# =========================
# TF-IDF WORD + CHAR N-GRAMS
# =========================
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    min_df=2,
    max_features=50000,
    lowercase=False,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=2,
    max_features=80000,
    lowercase=False,
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(X_train_texts)
X_val_word = word_vectorizer.transform(X_val_texts)

X_train_char = char_vectorizer.fit_transform(X_train_texts)
X_val_char = char_vectorizer.transform(X_val_texts)

X_train = hstack([X_train_word, X_train_char])
X_val = hstack([X_val_word, X_val_char])

print("\nShape X_train:", X_train.shape)
print("Shape X_val:", X_val.shape)

# =========================
# LINEARSVC
# =========================
model = LinearSVC(class_weight="balanced", C=0.5)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

print("\n===== VALIDATION RESULTS =====")
print("Validation accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification report:")
print(classification_report(y_val, y_pred, target_names=encoder.classes_))


joblib.dump(model, os.path.join(MODEL_DIR, "linear_svc_model.pkl"))
joblib.dump(word_vectorizer, os.path.join(MODEL_DIR, "word_vectorizer.pkl"))
joblib.dump(char_vectorizer, os.path.join(MODEL_DIR, "char_vectorizer.pkl"))
joblib.dump(encoder, os.path.join(MODEL_DIR, "label_encoder.pkl"))


Colonne train: ['model', 'text', 'topic', 'origin', 'length']
Shape train iniziale: (320593, 5)

Distribuzione classi:
target_label
Meta         86112
Anthropic    81438
Google       69388
Human        45053
OpenAI       33872
Name: count, dtype: int64

Classi finali usate dal modello:
0 -> Anthropic
1 -> Google
2 -> Human
3 -> Meta
4 -> OpenAI

Training samples: 252690
Validation samples: 63173

Shape X_train: (252690, 130000)
Shape X_val: (63173, 130000)

===== VALIDATION RESULTS =====
Validation accuracy: 0.9458154591360233

Classification report:
              precision    recall  f1-score   support

   Anthropic       0.93      0.91      0.92     16288
      Google       0.95      0.95      0.95     13878
       Human       0.99      0.99      0.99      9011
        Meta       0.93      0.94      0.93     17222
      OpenAI       0.97      0.99      0.98      6774

    accuracy                           0.95     63173
   macro avg       0.95      0.95      0.95     63173
weighted 

In [5]:
import joblib
import pandas as pd
from scipy.sparse import hstack


TEST_FILE = "subm3.csv"
MODEL_DIR = "models"
OUTPUT_FILE = "submission_subm3.csv"


model = joblib.load(f"{MODEL_DIR}/linear_svc_model.pkl")
word_vectorizer = joblib.load(f"{MODEL_DIR}/word_vectorizer.pkl")
char_vectorizer = joblib.load(f"{MODEL_DIR}/char_vectorizer.pkl")
encoder = joblib.load(f"{MODEL_DIR}/label_encoder.pkl")


df_test = pd.read_csv(TEST_FILE, sep=";", encoding="utf-8-sig")


id_col = "ID"
text_col = "Text"

df_test[text_col] = df_test[text_col].astype(str)

X_test_word = word_vectorizer.transform(df_test[text_col])
X_test_char = char_vectorizer.transform(df_test[text_col])
X_test = hstack([X_test_word, X_test_char])

pred_num = model.predict(X_test)
pred_labels = encoder.inverse_transform(pred_num)

submission = pd.DataFrame({
    "id": df_test[id_col],
    "label": pred_labels
})

submission.to_csv(OUTPUT_FILE, index=False)

print(submission.head())

Submission salvata: submission_subm3.csv
       id      label
0  D2-126  Anthropic
1  D2-127      Human
2  D2-128     OpenAI
3  D2-129      Human
4  D2-130  Anthropic
